# Experiment 20 — C00 versus deployed v11 rules

Based on [NB19](19-backtest-protected-core-score-risk-allocation.ipynb) and [NB15](15-backtest-allocation-vs-v11.ipynb).

**Goal:** compare C00 and hyper-ai-v11.py under identical data, capital, timing, fees and accounting. No parameter search. C00 changes the score, risk-estimation package and allocation method together; this experiment measures the combined strategy difference, not each change's isolated contribution.

**Periods:** 2026-01-01 → 2026-10-05 exclusive (through 4 October); full history 2023-03-01 → 2026-10-08 exclusive (through 7 October, last archive day partial). Fresh USD 150,000 in each account, all preceding warmup including 2025. 2026 outputs appear first, with full-history outputs explicitly labelled.

## Key new insights

2026: C00 changes profit by USD +9,630, CAGR by +9.44 percentage points and Sharpe by +0.260. The largest positive vault contribution to the difference is [ Systemic Strategies ] L/S Grids, USD +4,868 (50.5% of the net account gap).

Long: C00 changes profit by USD +143,804, CAGR by +14.58 percentage points and Sharpe by +0.278. The largest positive vault contribution to the difference is Road to 10 Million, USD +26,941 (18.7% of the net account gap).

C00 improves historical CAGR and Sharpe in both periods, supporting the economic hypothesis on these inspected samples. Full-history drawdown is worse, so this is not an improvement in every risk measure. In 2026, C00 earns more despite higher average cash. Full-history differences also include earlier deployment and different accumulated wealth. Isolating score, risk estimation and waterfall contributions remains a separate experiment.

## Summary of results

| Period | V11 CAGR | C00 CAGR | V11 Sharpe | C00 Sharpe | V11 max DD | C00 max DD | C00 extra profit |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 2026 | 49.15% | 58.59% | 2.214 | 2.474 | -6.41% | -6.36% | USD +9,630 |
| Long | 18.55% | 33.13% | 1.374 | 1.651 | -7.92% | -9.94% | USD +143,804 |

## Robustness of results

2026 C00: largest positive contributor Realist Capital accounts for 35.4% of positive vault profit; best daily update 6.46%, excess kurtosis 11.0. Long C00: largest positive contributor Delta_01 accounts for 17.2% of positive vault profit; best daily update 15.65%, excess kurtosis 90.9.

2026 C00 strongest update: Realist Capital contributes USD 6,419, with a 14.1% NAV change across 2.0 days; BTC and ETH change -4.7% and -6.0% over that observation interval. Curator excluded=False, quarantined=False.

Long C00 strongest update: Road to 10 Million contributes USD 15,536, with a 30.3% NAV change across 7.0 days; BTC and ETH change 19.6% and 17.1% over that observation interval. Curator excluded=False, quarantined=False.

C00 reproduces NB19's daily equity and metrics. All executed fills pass the as-of-NAV audit, and position profit reconciles to account profit within USD 1. Common partial-sell, cached-tax and funding corrections apply to both arms. Strongest vault updates are checked against curator flags and BTC/ETH over their actual observation intervals in the tables below. Sparse NAV may bunch returns; last-known NAV does not establish an executable live price. The two periods overlap and were previously researched; there is no untouched holdout or statistical significance claim. Universe/guard selection is retrospective and inherited fee assumptions remain economically unverified. No deployment-server revision was verified.


In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'clean_c00_v11_comparison.py').exists():
    ROOT=ROOT/'scratchpad/hyperliquid-vov-2nd'
sys.path.insert(0,str(ROOT))
from clean_c00_v11_comparison import initialise,run,report
from score_matrix_visualisation import compact_output


## Freeze shared inputs

Use the repaired archive and the same frozen strategy/framework identities as NB19. Retain all available warmup; no data downloads or parameter search.

In [2]:
with compact_output('Shared data and source identity; estimated 2–4 minutes'):
    data,evaluator=initialise()

Shared data and source identity; estimated 2–4 minutes — complete

## Execute four independent accounts

Replay V11 and C00 for 2026 and full history. Both start from cash under identical accounting corrections. Require C00 to reproduce NB19 before reporting comparative results.

In [3]:
with compact_output('Four full-engine replays; estimated 4–10 minutes'):
    results,runs,attribution,absolute_weights=run(data,evaluator)

Four full-engine replays; estimated 4–10 minutes — complete

## Compare outcomes and inspect robustness

Each period has side-by-side metrics, overlaid equity/drawdown/cash charts, separate absolute asset-weight charts and quarterly returns. Individual vault contributions and strongest-update market context explain concentrated results.

In [4]:
with compact_output('Charts, accounting and curator audit; estimated 1–3 minutes'):
    findings=report(data,results,runs,attribution,absolute_weights)

Charts, accounting and curator audit; estimated 1–3 minutes — complete

### Strategy logic — applies to both backtesting periods

These are strategy rules, not separate fitted configurations for each period. Scores select vaults; risk estimates determine allocation. The local v11 module represents deployed rules, not a verified replay of live account decisions.

| Rule | V11 | C00 |
| --- | --- | --- |
| Ranking | 0.60 CAGR360 + 0.40 Sortino45 | 0.48 CAGR360 + 0.32 Sortino45 + 0.10 CAGR180 + 0.10 Sortino30 |
| Component clips | CAGR 0–100%; Sortino 0–3, normalised to 0–1 | Same |
| Risk | Source 90-day variance, volatility floor 0.0001 | 90-day variance blended with leave-one-out positive peer median; own weight m/(m+30), m actual updates |
| Allocation | Proportional inverse variance with source capacity adjustment | Waterfall: lowest regularised risk first, filling each permitted capacity |
| Missing score | Composite becomes unscored/zero; eligible vaults can still enter | Same; no available-component reweighting |
| Missing risk | Source missing indicator gives zero raw signal | Partial history and peer prior supported; unusable basket risk falls back to equal raw priorities |
| Maximum positions | 6; no capacity extension | Same |
| Selection | Source rank hysteresis, entry 6 / exit 8, minimum improvement 0 | Same machinery, different score values |
| Limits | 98% deployment, 33% concentration, 33% vault TVL, minimum TVL USD 7,500 | Same |
| Rebalance and gate | Every 2 days; 14-day return must exceed −16% | Same |
| Execution thresholds | Buy 0.05%, sell 0.5% of reference equity, USD 5 floor; cleanup 0.5% | Same |
| Permissions | Source guard, whitelist, deposit/redemption and leader-share policy | Same |

### Common accounting and data — applies to both backtesting periods

Fresh USD 150,000 accounts; the same repaired archive, universe, guard and preceding warmup, including 2025 for the 2026 accounts. Indicators use completed prior daily buckets. Execution uses the latest recorded NAV at or before each query, with no additional slippage or freshness veto. Causal research NAV filling starts at inception; missing observations are not backfilled from the future.

Both arms use three identical research accounting corrections: partial sells charge the modelled redemption fee once against gross observed NAV; newly reopened positions reset the previous position's cached tax before marking; buys are bounded by final synchronous net sell proceeds. These changes leave ranking and sizing rules intact. Modelled charges remain 10% of positive redemption profit plus a 0.1% capital allowance; the allowance is an assumption, not a verified venue fee. Live v11 pays actual execution costs, so this is a clean comparison of rules rather than a live-performance replication.


## 2026 comparison

2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC). Independent fresh $150,000 accounts.

### Key metrics — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

Green marks the better result within each economic metric. Daily-resampled risk metrics use 365-day annualisation, including non-decision days; both strategies use the same calculation.

Arm,V11,C00
Final equity (USD),"$202,746","$212,376"
Net profit (USD),"$52,746","$62,376"
Cumulative return,35.16%,41.58%
CAGR,49.15%,58.59%
Sharpe,2.214,2.474
Sortino,3.953,4.532
Calmar,7.673,9.215
Maximum drawdown,-6.41%,-6.36%


...

Arm,V11,C00
Paid sell deductions (USD),"$34,334.65","$56,372.40"
Open performance liability (USD),"$1,598.62","$1,646.34"
Open capital allowance (USD),$70.74,$72.88
Gross NAV equity (USD),"$278,490.09","$422,582.79"
Estimated liquidation equity (USD),"$276,820.72","$420,863.57"
Reported marked equity (USD),"$277,077.90","$420,881.97"
Open marks vs net estimate (USD),$257.18,$18.41


### C00 replication against NB19 — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

Both complete daily equity paths and key metrics must match the latest completed NB19. V11 is independently replayed with the same common accounting corrections.

Period,Maximum equity difference (USD),Saved NB19 metrics match
2026,$0.000000000,True
Long,$0.000000000,True


## Findings

2026: C00 changes profit by USD +9,630, CAGR by +9.44 percentage points and Sharpe by +0.260. The largest positive vault contribution to the difference is [ Systemic Strategies ] L/S Grids, USD +4,868 (50.5% of the net account gap).

Long: C00 changes profit by USD +143,804, CAGR by +14.58 percentage points and Sharpe by +0.278. The largest positive vault contribution to the difference is Road to 10 Million, USD +26,941 (18.7% of the net account gap).

<details><summary>Data, source and framework identity</summary>

```json
{
  "identity": {
    "version": "nb20-v1",
    "parent": {
      "version": "nb19-v1",
      "inputs": {
        "version": "nb17-v1",
        "nb16_inputs": {
          "version": "nb15-v1",
          "nb14": {
            "version": "nb14-v1",
            "code": {
              "profitable_allocation.py": "a110b6debb07f9961f13e3ff56f9be634bacfd75ec9f98e3e7c8c3f7279d5a88",
              "profitable_allocation_report.py": "4b2587dfec7419fca290f598cca2cea1a64a85c4f2dcd20a5356d5d2cf9c255f",
              "create_profitable_allocation_notebook.py": "b41c7e4b2070eccf79e784f9df61b38d37f3b407b0969875563374a9dbac92ca"
            },
            "nb13_identity": {
              "source_fingerprint": "ea6a622a7706d49ddb818700e6b3d30ccdb412c41bfe0db2c9d1187515c44cdf",
              "extended_universe_fingerprint": "67bffc9ed7d5997250513dd6fba3e00cbc275f91ed65300664a6cc31c49fad1e",
              "feature_panel": "f7e29b5e36a30531933bc2e188dfa30c99e29253b3ccd7071e84f1b0dee35b73",
              "code": {
                "long_candidate_backtest.py": "1c60331cab792097a436f2e2c0f1d8c6e87f98aabc51a1a633e3c3d98347b1ff",
                "score_matrix.py": "e3ca6d6bf07f418121dabf91aa6e21d5a9eeaf5cc4bb86120c233e4db089a503",
                "allocation_followup.py": "eb2513a3d8d0e2e7d6d0cc447c700705d3ec1f01a954d462c898288898d07b87",
                "allocation_followup_engine.py": "0951f17d4c7ce3af6f3bca2b361d439057535ae12c0433ae006c772aae425048",
                "portfolio_snapshot_risk.py": "d6a10069cf851d9504eb50fd6698cd05fd3de109d8a91ad76d6ffada4c73b854",
                "sizing_study_data.py": "fa07af5ed0b936be2fa43ebe410858d28295673554cff0fdda7cedcbded33b1f"
              },
              "shortlist": [
                {
                  "Candidate": "C01",
                  "Name": "Legacy 45d / 4 inputs / 6m Sortino + 2m CAGR / 6 positions",
                  "Positions": 6.0,
                  "Kind": "score",
                  "Score name": "Legacy 45d / 4 inputs / 6m Sortino + 2m CAGR",
                  "Policy": ""
                },
                {
                  "Candidate": "C02",
                  "Name": "Blend 46d / 3 inputs / 6m CAGR / 6 positions",
                  "Positions": 6.0,
                  "Kind": "score",
                  "Score name": "Blend 46d / 3 inputs / 6m CAGR",
                  "Policy": ""
                },
                {
                  "Candidate": "C03",
                  "Name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino / 6 positions",
                  "Positions": 6.0,
                  "Kind": "score",
                  "Score name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino",
                  "Policy": ""
                },
                {
                  "Candidate": "C04",
                  "Name": "Blend 46d / 7 positions",
                  "Positions": 7.0,
                  "Kind": "score",
                  "Score name": "Blend 46d",
                  "Policy": ""
                },
                {
                  "Candidate": "C05",
                  "Name": "Legacy 45d / 4 inputs / 2m CAGR + 6m Sharpe / 6 positions",
                  "Positions": 6.0,
                  "Kind": "score",
                  "Score name": "Legacy 45d / 4 inputs / 2m CAGR + 6m Sharpe",
                  "Policy": ""
                },
                {
                  "Candidate": "C06",
                  "Name": "Blend 46d / 4 inputs / 6m Sortino + 2m CAGR / 7 positions",
                  "Positions": 7.0,
                  "Kind": "score",
                  "Score name": "Blend 46d / 4 inputs / 6m Sortino + 2m CAGR",
                  "Policy": ""
                },
                {
                  "Candidate": "C07",
                  "Name": "NB12 S18 F:gap_log L90/p2/\u03ba30",
                  "Positions": 6.0,
                  "Kind": "allocation",
                  "Score name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino",
                  "Policy": "S18"
                },
                {
                  "Candidate": "C08",
                  "Name": "NB12 X01 F:regular L75/p2/\u03ba30",
                  "Positions": 6.0,
                  "Kind": "allocation",
                  "Score name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino",
                  "Policy": "X01"
                },
                {
                  "Candidate": "C09",
                  "Name": "Legacy 45d / 4 inputs / 2m CAGR + 6m Sharpe / 7 positions",
                  "Positions": 7.0,
                  "Kind": "score",
                  "Score name": "Legacy 45d / 4 inputs / 2m CAGR + 6m Sharpe",
                  "Policy": ""
                },
                {
                  "Candidate": "C10",
                  "Name": "NB12 S17 F:calendar_log L90/p2/\u03ba30",
                  "Positions": 6.0,
                  "Kind": "allocation",
                  "Score name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino",
                  "Policy": "S17"
                }
              ],
              "shortlist_data": {
                "07/2026-results.csv": "860cbacb9fa78a6281276d72a28f9ab08c18610d12368efb8c422d59e6f8ff1f",
                "12/465e3b8cf13b49e11f635d12c951268318e3b2c39e120f25b89190dd66606be5/engine_summary.parquet": "bb8fabffd0d61bfe4eab10a392873e49cf3fd4a1661439f41082247371909805"
              },
              "downloads": {
                "vault_prices": {
                  "path": "/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd/_artifacts-restored-2026-10-07/inputs/vault-prices.parquet",
                  "sha256": "30ec7e9dfa7eb236518cf1922a9ba82f4f42cfbfb041c06f5db3273d114aa8f2",
                  "bytes": 289315157
                },
                "vault_metadata": {
                  "path": "/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd/_artifacts-restored-2026-10-07/inputs/vault-metadata.json",
                  "sha256": "6c5ec12de41f1c10fc48e54a832eff065d14c77ea19c85fd482c46ea33a5382c",
                  "bytes": 83587673
                },
                "permission_sidecar_status": {
                  "http_status": 404,
                  "used": false,
                  "note": "Source strategy uses permission projection; independent endpoint returned 404"
                }
              },
              "start": "2023-03-01 00:00:00",
              "end": "2026-10-08 00:00:00",
              "control_arms": [
                "v11",
                "regular90"
              ]
            },
            "arms": {
              "A": [
                "Source variance / original allocation",
                "source",
                "sequential"
              ],
              "B": [
                "Source variance / simultaneous allocation",
                "source",
                "simultaneous"
              ],
              "C": [
                "Regularised variance / original allocation",
                "regular",
                "sequential"
              ],
              "D": [
                "Regularised variance / simultaneous allocation",
                "regular",
                "simultaneous"
              ],
              "G": [
                "Regularised downside / original allocation",
                "downside",
                "sequential"
              ]
            },
            "periods": {
              "2026": [
                "2026-01-01 00:00:00",
                "2026-10-05 00:00:00"
              ],
              "Long": [
                "2023-03-01 00:00:00",
                "2026-10-08 00:00:00"
              ]
            },
            "pricing": "as_of_observation",
            "sensitivity": "fresh_quote_within_source_two_day_cycle",
            "initial_cash": 150000
          },
          "source": "/Users/moo/code/strategies/strategy/hyper-ai-v11.py",
          "source_sha256": "5ed8a4c00f1edb14ae9af40ac3d94d3622587445921d1e92ef33f9676761ffbe",
          "code": {
            "allocation_comparison.py": "a7b804e4e4d214d91eee77ee764f914fee998143777065c25809bddaa6db4c76",
            "create_allocation_comparison_notebook.py": "e603ce4edfdf364699177337507baafacef319e4a3777f923a2eba0873430081"
          },
          "arms": [
            "V11",
            "A",
            "C",
            "G"
          ],
          "initial_cash": 150000,
          "periods": {
            "2026": [
              "2026-01-01 00:00:00",
              "2026-10-05 00:00:00"
            ],
            "Long": [
              "2023-03-01 00:00:00",
              "2026-10-08 00:00:00"
            ]
          },
          "pricing": "as_of_observation"
        },
        "arms": {
          "M00": "Exact C-W benchmark",
          "M01": "Admission-aware risk waterfall",
          "M02": "Proportional capped risk",
          "M03": "Score-priority waterfall",
          "M04": "Core + two score-first overflow",
          "M05": "Core + four score-first overflow",
          "M06": "Core + four scored-only overflow",
          "M07": "Capacity-aware incumbent retention",
          "M08": "Greedy score-dollar basket",
          "M09": "Maintained / fee-aware overflow",
          "M10": "One-decision rank-exit confirmation",
          "M11": "Joint six-slot score optimiser",
          "M12": "Legacy capacity extension",
          "M13": "Fixed ten-name core"
        },
        "numerical": {
          "dollar_tolerance": 1e-06,
          "objective_tolerance": 1e-10,
          "mip_rel_gap": 1e-08,
          "time_limit": 30.0
        },
        "code": {
          "capital_allocation.py": "7e4b90c272d628eb3a114aaa0efeff93e18972df0914a9fa818dcc77069a27ab",
          "capital_allocation_methods.py": "9c54723c01be9a0ada347842ddb254717b032703afc02f3ba2c6bcc5aad234b3"
        },
        "source_sha256": "5ed8a4c00f1edb14ae9af40ac3d94d3622587445921d1e92ef33f9676761ffbe"
      },
      "methods": {
        "C00": "Exact corrected C-W",
        "E00": "Flat tickets / hold units",
        "R00": "Risk tickets / hold units",
        "S00": "Score tickets / hold units",
        "RS00": "Score \u00d7 risk / hold units",
        "RS01": "Score \u00d7 risk / resize",
        "NRS00": "Normalised score \u00d7 risk / resize",
        "RS08": "Score \u00d7 risk / eight slots",
        "RS12": "Score \u00d7 risk / twelve slots",
        "GF25": "Score floor 0.25 / hold",
        "CAP10": "Smaller exposure caps / hold"
      },
      "registry": {
        "risk_days": 90,
        "actual_update_shrinkage": 30,
        "score_floor": 0.5,
        "single_gross_cap": 0.05,
        "total_gross_cap": 0.25,
        "ticket_fraction": 0.5,
        "core_slots": 6,
        "entry_rank": 6,
        "exit_rank": 8,
        "missing": true,
        "initial_cash": 150000
      },
      "code": "ad815fcb39ab3bac775434f02bf6fff5429dce2efabd1b08ef804c5a926f76cc"
    },
    "code": "45455be53b9201d10d71e198f3890dd92e2dc72f8b6bbfca5c62eee7b4cb3644",
    "arms": [
      "V11",
      "C00"
    ],
    "accounting": "single partial-sell fee, fresh-entry tax reset, final-proceeds funding",
    "reference": "/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd/_artifacts-restored-2026-10-07/19/4cf096b0443683cc5ebffa060b3756829c64a636160ae9c3b4d6b1a5f81ab6b4"
  },
  "framework": {
    "getting-started": "0be3392775ffd2567a617153a3c8d42fb0e57476",
    "trade-executor": "2db909b32d854d342f159074b2b92948f32b69d6",
    "trading-strategy": "unavailable",
    "web3-ethereum-defi": "unavailable"
  },
  "periods": {
    "2026": [
      "2026-01-01 00:00:00",
      "2026-10-05 00:00:00"
    ],
    "Long": [
      "2023-03-01 00:00:00",
      "2026-10-08 00:00:00"
    ]
  },
  "limitations": [
    "Researched overlapping histories, retrospective universe/guard",
    "NAV observation timestamp assumed available; executable NAV unverified",
    "Inherited 10% profit and 0.1% capital fee allowances",
    "Same-state core intentions are protected; future wealth and core membership can change"
  ],
  "fingerprint": "43eea0e427a4571a73b4ac32c50bb29ecbdac66e8542dd4573a6401182641911"
}
```

</details>